# P5 — Task 5 — Recommendations + ráp deck + nộp bài (Lead)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanh-tai-435/BI10GroupYAPPERS/blob/main/notebooks/task5_recommendations.ipynb)

**Slide sở hữu:** 1, 2, 3, 16, 17, 18 · **Slide mới:** +1: Tác động dự kiến & lộ trình  
**Đã có:** task5.py, task5_findings.md, chart t5_priority_ranking / t5_target_sizes, build_deck.js, YAPPERS_BI10_R01.pptx

**3 luật:** (1) Wellbeing, KHÔNG credit scoring — FHS không bao giờ dùng để từ chối/giảm hạn mức/khoá. (2) So bằng tỷ lệ, không so VND tuyệt đối. (3) Mỗi nhận định có số hoặc chart.

Code đầy đủ: `draft/task5_deep.py` (chạy độc lập được). Hướng dẫn chi tiết: `draft/YAPPERS_huong_dan_Colab.xlsx`, sheet của bạn.

In [ ]:
#@title ⚙️ SETUP — chạy 1 lần mỗi phiên Colab (không cần pip install gì cả)
# Colab có sẵn pandas / numpy / scikit-learn / matplotlib / openpyxl -> đủ cho mọi script trong draft/.
import os, re, sys, gzip, shutil, urllib.request
REPO = "/content/BI10GroupYAPPERS"
# Data tải thẳng từ thư mục Drive public (anyone with link) — không cần mount Drive / Add shortcut
# Thư mục: https://drive.google.com/drive/folders/1gOqCyY9IfFUJtbr2r3vR_Z3t61O71CrC
URL = {
    "consumer_financial_health_engagement_2025.csv": "https://drive.google.com/file/d/1YyueAH9Kasid6V_obml9pCMwh2dB0ryn/view",
    "consumer_transactions_2025.parquet.gz":         "https://drive.google.com/file/d/1O33KpFbUx6_BRGwFTb80xVSB3tzXqt5H/view",
    "data_dictionary.xlsx":                          "https://drive.google.com/file/d/1nFlZHmuaoK8VQ5xqRN2f4ZLqoR3gk2j1/view",
}

from google.colab import userdata

if os.path.exists(REPO):
    os.system(f"git -C {REPO} pull -q")
else:
    try:
        tok = userdata.get("GH_TOKEN")   # 🔑 Secrets bên trái: tên GH_TOKEN, bật "Notebook access"
    except Exception:                    # chưa tạo secret -> hỏi trực tiếp (không lưu vào notebook)
        from getpass import getpass
        tok = getpass("Chưa có secret GH_TOKEN — dán GitHub token (ghp_...) rồi Enter: ").strip()
    os.system(f"git clone -q https://{tok}@github.com/thanh-tai-435/BI10GroupYAPPERS.git {REPO}")
assert os.path.exists(REPO), "Clone thất bại: kiểm tra GH_TOKEN + đã được mời vào repo chưa"
os.chdir(REPO)

# Dựng lại đúng đường dẫn mà các script draft/*.py đang dùng
D = "BI10_ROUND01_DATASET"
TX = f"{D}/consumer_transactions_2025.parquet/consumer_transactions_2025.csv"
os.makedirs(os.path.dirname(TX), exist_ok=True)
def fetch(u, dst, tries=3):   # tải vào .part rồi mới đổi tên -> không bao giờ để lại file dở
    fid = re.search(r"/d/([\w-]+)", u).group(1)   # tải file public bằng stdlib, không cần gdown
    for i in range(1, tries + 1):
        try:
            urllib.request.urlretrieve(f"https://drive.usercontent.google.com/download?id={fid}&export=download&confirm=t", dst + ".part")
            return os.replace(dst + ".part", dst)
        except Exception as e:
            print(f"Tải {os.path.basename(dst)} lỗi ({type(e).__name__}), thử lại {i}/{tries}...")
    raise RuntimeError(f"Không tải được {dst}: kiểm tra mạng / thư mục Drive còn quyền 'Anyone with the link', rồi Run all lại")
for f, u in URL.items():
    if not os.path.exists(f"{D}/{f}"):
        fetch(u, f"{D}/{f}")
if not os.path.exists(TX):   # .parquet.gz thực chất là CSV giao dịch nén gzip (134MB -> 597MB, ~30s)
    with gzip.open(f"{D}/consumer_transactions_2025.parquet.gz") as src, open(TX + ".part", "wb") as dst:
        shutil.copyfileobj(src, dst)
    os.replace(TX + ".part", TX)

# Script gốc gọi sys.stdout.reconfigure(); stdout của Colab không có hàm này -> cho thành no-op
if not hasattr(sys.stdout, "reconfigure"):
    type(sys.stdout).reconfigure = lambda self, **k: None
os.makedirs("outputs/figures", exist_ok=True)
if not os.path.exists("draft/task4_features.csv"):   # nhãn segment (git-ignore) — task3/task5/charts cần
    import subprocess   # sys.executable = đúng Python của notebook (có pandas)
    subprocess.run([sys.executable, "draft/task4.py"], check=True, capture_output=True,
                   env={**os.environ, "PYTHONUTF8": "1"})
print("✅ Sẵn sàng — cwd:", os.getcwd())


## Bước 0 — Chạy baseline
Nạp toàn bộ biến của script gốc (`mon`, `cons`, `txn`…) vào notebook. So kết quả với `draft/task5_output.txt`.

In [ ]:
%run -i draft/task5.py

In [ ]:
# Style chung cho chart (giống draft/make_charts.py)
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 250, "display.max_columns", 30)
plt.rcParams.update({"figure.figsize": (10, 6), "font.size": 11, "axes.titlesize": 15, "axes.titleweight": "bold",
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": .25})
C = dict(primary="#2E5EAA", accent="#E4572E", good="#3C9D6E", warn="#E0A32E", muted="#9AA5B1", ink="#22303F")
os.makedirs("outputs/figures", exist_ok=True)
def save(fig, name):
    fig.tight_layout(); fig.savefig(f"outputs/figures/{name}", dpi=150, bbox_inches="tight"); plt.show(); print("saved", name)
def h(t): print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78)


## Việc 1 — Setup chung cho cả nhóm (LÀM ĐẦU TIÊN, 26/09 tối)  `[BẮT BUỘC · hạn 26/09 22:00]`
Xem sheet 'Setup Colab' phần Leader: push code, mời 4 bạn vào repo, share thư mục data trên Drive.

**Giao:** Cả nhóm mở được notebook

In [ ]:
# Việc không cần code phân tích — gợi ý:
# —


## Việc 2 — Slide mới 'Tác động dự kiến & lộ trình'  `[Cao · hạn 27/09 18:00]`
Hồi quy đơn FHS ~ spend_to_income (cấp tháng). Nếu Budgeting giúp 302 khách giảm spend/income 5% / 10% → FHS tăng bao nhiêu, bao nhiêu tháng thoát FHS<40? Ghi rõ KỊCH BẢN GIẢ ĐỊNH. Mỗi công cụ 1 KPI + đo bằng A/B test. Lộ trình 30/60/90 ngày.

**Giao:** Slide mới

In [ ]:
h("Impact scenario - what if Budgeting trims spend-to-income? (illustrative, NOT a forecast)")
b, a = np.polyfit(m.spend_to_income_ratio, m.financial_health_score, 1)
r = m.spend_to_income_ratio.corr(m.financial_health_score)
print(f"Month-level fit: FHS = {a:.1f} {b:+.2f} x spend_to_income  (r = {r:.2f}, n = {len(m):,})")
stretched_ids = set(f.loc[f.segment == "Financially Stretched but Highly Engaged", "consumer_id"])
ms = m[m.consumer_id.isin(stretched_ids)]
rows = []
for cut in (0, .05, .10, .15):
    fhs_new = m.financial_health_score - b * m.spend_to_income_ratio * cut * m.consumer_id.isin(stretched_ids)
    st_new = fhs_new < 40
    rows.append(("Today" if cut == 0 else f"-{int(cut*100)}%", round(float((ms.financial_health_score - b * ms.spend_to_income_ratio * cut).mean()), 1),
                 int(st_new.sum()), int(m.loc[st_new, "consumer_id"].nunique())))
sc = pd.DataFrame(rows, columns=["spend/income change (Stretched 302)", "Stretched mean FHS", "stress months (all 999)", "consumers with a stress month"])
print(sc.to_string(index=False))
base_st = sc.iloc[0, 2]
for i in (1, 2):
    print(f"  {sc.iloc[i,0]}: stress months {base_st} -> {sc.iloc[i,2]} ({100*(sc.iloc[i,2]/base_st-1):+.0f}%)")
print("Caveat: FHS is partly built from spend fields -> slope is mechanical, directional only; validate with an A/B test.")

fig, ax = plt.subplots(figsize=(9, 5.5))
bars = ax.bar(sc.iloc[:, 0], sc.iloc[:, 2], color=[C["accent"], C["warn"], C["good"], C["good"]])
for rct, v in zip(bars, sc.iloc[:, 2]):
    ax.annotate(f"{v}", (rct.get_x() + rct.get_width()/2, v), xytext=(0, 3), textcoords="offset points", ha="center", fontsize=12)
ax.set_xlabel("Assumed change in spend-to-income for the Stretched segment (302)")
ax.set_ylabel("Consumer-months with FHS < 40 (all 999)")
ax.set_title("Scenario: a 10% spend trim cuts stress months by a third")
save(fig, "t5_impact_scenario.png")


In [ ]:
h("Measurement plan - one KPI per tool, each tested with a randomised holdout")
kpi = pd.DataFrame([
    ("Budgeting tool", "Stretched & Engaged (302)", "spend-to-income ratio; % months FHS<40", "10% random holdout, 3 months"),
    ("Spend alerts", "Crossover (43; 52% of stress months)", "stress months per consumer; discretionary share in alert window", "alert vs no-alert, 2 months"),
    ("Planning reminders", "Power Users (258) + Healthy (351) before Dec", "Nov->Dec slide Healthy->Stretched (base 72.4%)", "reminder in Nov vs holdout"),
    ("Financial education", "Distress tail (67)", "module completion; essential share next month", "opt-in cohort vs matched"),
    ("Digital nudges", "Emerging Digital (88)", "active days / month; category diversity", "nudge vs holdout, 2 months"),
    ("Product suggestions", "Healthy & Engaged (351)", "opt-in savings/loyalty uptake; FHS stable", "offer vs holdout; guardrail: FHS never used for credit"),
], columns=["Tool", "Target", "KPI", "Test design"])
print(kpi.to_string(index=False))


In [ ]:
h("30/60/90-day roadmap")
for k, v in {"Days 0-30": "Launch Budgeting (302) + Spend Alerts (43), timed Sun-Mon & 22-23h (Task 2 timing); set holdouts",
             "Days 31-60": "Planning Reminders before December (Nov->Dec slide 72.4%); Digital Nudges (88)",
             "Days 61-90": "Education (67) + opt-in Product suggestions (351); read A/B results, keep what moves the KPI"}.items():
    print(f"  {k:<11} {v}")


## Việc 3 — Cập nhật 6 đề xuất với input đồng đội  `[Cao · hạn 27/09 20:00]`
Giờ/ngày gửi alert (P2), % trượt segment T11–12 (P4), ai dùng QR/Mobile (P3).

**Giao:** Slide 16–17

In [ ]:
# Việc không cần code phân tích — gợi ý:
# —


## Việc 4 — Luật tín dụng tuyệt đối  `[BẮT BUỘC · hạn 28/09 12:00]`
Đọc lại toàn deck, xoá mọi câu có thể hiểu là dùng FHS cho quyết định tín dụng.

**Giao:** Slide 17

In [ ]:
# Việc không cần code phân tích — gợi ý:
# —


## Việc 5 — Ráp deck (chạy LOCAL, không phải Colab)  `[BẮT BUỘC · hạn 27/09 23:00]`
Người DUY NHẤT sửa build_deck.js. Chèn 4 slide mới: sau slide 7 (P1), sau 10 (P2), sau 15 (P4), trước Closing (P5). Cập nhật mục lục slide 3.

**Giao:** Deck v2 (22 slide) + PDF gửi nhóm

In [ ]:
# Việc không cần code phân tích — gợi ý:
# PYTHONUTF8=1 py draft/make_charts.py && node draft/build_deck.js
# "/c/Program Files/LibreOffice/program/soffice.exe" --headless --convert-to pdf --outdir draft/ draft/YAPPERS_BI10_R01.pptx


## Việc 6 — QA + soát nhất quán số  `[BẮT BUỘC · hạn 28/09 18:00]`
Nhận danh sách lệch số từ P3. ≤22 slide, 16:9, tiếng Anh, chart có caption/nguồn, không chữ tràn.

**Giao:** Deck final

In [ ]:
# Việc không cần code phân tích — gợi ý:
# pdftoppm -png -r 100 draft/YAPPERS_BI10_R01.pdf draft/slide


## Việc 7 — Nộp bài  `[BẮT BUỘC · hạn 28/09 20:00]`
Đổi tên YAPPERS_<TênLeader>_BI10_R01.pptx. ZIP YAPPERS_<TênLeader>_BI10_R01_data.zip: draft/*.py, *_output.txt, *_findings.md, notebooks/, task4_features.csv + dictionary, outputs/figures/.

**Giao:** Nộp

In [ ]:
# Việc không cần code phân tích — gợi ý:
# —


## Findings — kết quả đã chạy 27/09 (chi tiết: `draft/task5_deep_output.txt`)
- Kịch bản (giả định): FHS = 84.1 − 25.3 × spend/income (r −0.90). Stretched (302) giảm spend/income **5% → tháng stress 95→80 (−16%)**, **10% → 65 (−32%)**, 15% → 49.
- KPI + A/B holdout cho cả 6 công cụ; lộ trình 30/60/90 ngày: Budgeting + Alerts (tối CN–T2) → Reminders trước T12 + Nudges → Education + Products.
- Input đồng đội đã đưa vào: giờ gửi alert (P2), 72.4% trượt T11→T12 (P4), Emerging Digital dùng Mobile/E-com 60.5% giá trị (P3).

---
**Lưu bài:** `File → Save a copy in GitHub` → repo `thanh-tai-435/BI10GroupYAPPERS`, branch `main`, path `notebooks/task5_recommendations.ipynb` (mỗi người chỉ lưu notebook của mình → không conflict). Chart: `savefig('outputs/figures/tN_ten.png')` rồi tải về gửi P5.